# Module 2 Exercises


In [ ]:
import os
from typing import List, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from agents import (
    Agent,
    Runner,
    trace,
    function_tool,
    OpenAIChatCompletionsModel,
    input_guardrail,
    output_guardrail,
    GuardrailFunctionOutput,
)


In [ ]:
load_dotenv(override=True)

def debug_print(*args, **kwargs):
    """Print to console only if DEV_ENV is set to 'development'."""
    if os.environ.get("DEV_ENV", "").lower() == "development":
        print(*args, **kwargs)

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

debug_print("OPENROUTER_KEY set:", bool(openrouter_api_key))


In [ ]:

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

def make_chat_model(model_name: str, base_url: str, api_key: str):
    """Return chat model"""
    client = AsyncOpenAI(base_url=base_url, api_key=api_key)
    return OpenAIChatCompletionsModel(model=model_name, openai_client=client)


models = {
    "openai": "openai/gpt-4o",
    "gemini": "google/gemini-2.0-flash-001",
    "deepseek": "deepseek/deepseek-chat",
    "anthropic": "anthropic/claude-3.5-sonnet",
}

model_registry = {
    "openai": make_chat_model(models["openai"], OPENROUTER_BASE_URL, openrouter_api_key),
    "gemini": make_chat_model(models["gemini"], OPENROUTER_BASE_URL, openrouter_api_key),
    "deepseek": make_chat_model(models["deepseek"], OPENROUTER_BASE_URL, openrouter_api_key),
    "anthropic": make_chat_model(models["anthropic"], OPENROUTER_BASE_URL, openrouter_api_key),
}


In [ ]:

class ColdEmail(BaseModel):
    """Structured email draft."""

    subject: str = Field(min_length=8, max_length=120)
    preview_text: str = Field(min_length=12, max_length=160)
    body_text: str = Field(min_length=80)
    cta: str = Field(min_length=8, max_length=120)
    tone: Literal["serious", "engaging", "concise", "playful"]


class NameCheckOutput(BaseModel):
    """Personal name detection output."""

    is_name_in_message: bool
    name: str


class SafetyReviewOutput(BaseModel):
    """Output policy review result."""

    blocked: bool
    reason: str


class Contact(BaseModel):
    """Contact information."""
    name: str
    email: str
    company: str


class EmailPayload(BaseModel):
    """Message envelope for a send operation."""
    to: str
    subject: str
    html: str


class ReviewResult(BaseModel):
    """Structured output for the review agent's draft selection."""
    chosen_tone: Literal["serious", "engaging", "concise", "playful"]
    justification: str = Field(description="A brief explanation of why this draft is the best fit for the campaign goals.")
    winning_draft_content: str = Field(description="The full Text of the winning draft.")

In [ ]:

SALES_CONTEXT = """
You work at ComplAI, an AI tool that helps teams prepare for SOC2 audits.
Write outbound emails to B2B prospects. Keep claims realistic and specific.
""".strip()

def make_sales_agent(name: str, tone: str, model):
    """Create one model-specific sales drafter."""
    instructions = f"""
{SALES_CONTEXT}
You write in a {tone} style.
Return output in the schema fields for subject, preview_text, body_text, cta, and tone.
""".strip()
    return Agent(name=name, instructions=instructions, model=model, output_type=ColdEmail)





concise_sales_agent = make_sales_agent("Concise Sales Agent", "concise", model_registry["deepseek"])
engaging_sales_agent = make_sales_agent("Engaging Sales Agent", "engaging", model_registry["anthropic"])
playful_sales_agent = make_sales_agent("Playful Sales Agent", "playful", model_registry["gemini"])
serious_sales_agent = make_sales_agent("Serious Sales Agent", "serious", model_registry["openai"])


In [ ]:
review_instructions = """
You are the Quality Assurance Director for the email campaign.
Your job is to review multiple candidate email drafts and select the best one based on:
1. Relevance to the specific campaign goals.
2. Clarity and professional tone.
3. Effectiveness of the Call-To-Action (CTA).

You will be provided with the drafts generated by different sales agents.
Evaluate them carefully and return your structured decision identifying the winning draft.
""".strip()

review_agent = Agent(
    name="review_agent",
    instructions=review_instructions,
    model=model_registry["openai"],
    output_type=ReviewResult,
)

In [ ]:
def _render_email_logic(email: ColdEmail, recipient_name: str, company: str) -> str:
    """The actual rendering logic, safe to call from other functions."""
    return f"""
    <html>
      <body style='font-family: Arial, sans-serif; max-width: 680px;'>
        <p>Hi {recipient_name},</p>
        <p>{email.body_text}</p>
        <p><strong>{email.cta}</strong></p>
        <p>Best,<br/>ComplAI Team</p>
        <hr/>
        <small>For {company} | Preview: {email.preview_text}</small>
      </body>
    </html>
    """.strip()


@function_tool
def get_target_contacts(segment: str = "soc2_readiness") -> List[Contact]:
    """Return a small target contact list."""
    debug_print(f"DEBUG: get_target_contacts called for segment: {segment}")
    data = {
        "soc2_readiness": [
            {
                "name": "Head of Engineering",
                "email": "eng-lead@example.com",
                "company": "Northwind",
            },
            {
                "name": "VP Security",
                "email": "security-vp@example.com",
                "company": "Contoso",
            },
            {
                "name": "Compliance Director",
                "email": "compliance@example.com",
                "company": "Fabrikam",
            },
        ]
    }
    contacts = data.get(segment, [])
    debug_print(f"DEBUG: Found {len(contacts)} contacts: {contacts}")
    return contacts


@function_tool
def render_html_email(email: ColdEmail, recipient_name: str, company: str) -> str:
    """Render HTML from structured fields."""
    debug_print(f"DEBUG: render_html_email called for {recipient_name} at {company}")
    return _render_email_logic(email, recipient_name, company)


@function_tool
def build_mail_merge_plan(
    email: ColdEmail, contacts: List[Contact]
) -> List[EmailPayload]:
    """Build recipient-specific payloads."""
    debug_print(f"DEBUG: build_mail_merge_plan started for {len(contacts)} contacts...")
    debug_print(f"DEBUG: Using email subject: {email.subject}")
    plan = []
    for contact in contacts:
        payload = EmailPayload(
            to=contact.email,
            subject=email.subject,
            html=_render_email_logic(email, contact.name, contact.company),
        )
        plan.append(payload)
        debug_print(f"DEBUG: Built payload for {contact.email}")

    debug_print(f"DEBUG: Successfully built {len(plan)} payloads.")
    return plan


@function_tool
def send_mail_merge_dry_run(payloads: List[EmailPayload]) -> str:
    """Dry run send operation that returns a formatted report."""
    debug_print(f"DEBUG: send_mail_merge_dry_run called with {len(payloads)} payloads.")

    report = f"### Dry Run Complete: {len(payloads)} Emails Prepared\n\n"
    for i, p in enumerate(payloads):
        report += f"**Recipient {i + 1}:** {p.to}\n"
        report += f"**Subject:** {p.subject}\n"
        report += f"**Content Preview:** {p.html[:150].strip()}...\n"
        report += "---\n\n"

        print(f"DEBUG: Payload {i + 1} to {p.to}:")
        print(f"DEBUG:   Subject: {p.subject}")

    # My sendgrid is unavailable atm
    debug_print("DEBUG: Successfully generated dry-run report.")
    return report


In [ ]:
name_guardrail_agent = Agent(
    name="name_guardrail",
    instructions=(
        "Check if the input asks to include a personal person name or direct private identifier. "
        "Set is_name_in_message=True ONLY when a specific personal first/last name is requested. "
        "Professional roles, job titles, or departments (e.g., 'CTO', 'Head of Engineering', 'Startup CTO') "
        "are NOT personal names and should NOT be blocked."
    ),
    output_type=NameCheckOutput,
    model=model_registry["openai"]
)


output_guardrail_agent = Agent(
    name="output_guardrail",
    instructions=(
        "You are a Safety Auditor. Review the human-readable text of the proposed email. "
        "IGNORE technical formatting, JSON structures, or debug logs. "
        "ONLY block if the actual email body contains: "
        "1. Deceptive marketing claims (e.g., '100% guaranteed SOC2 in 1 day'). "
        "2. Requests for sensitive passwords or credentials. "
        "3. Explicitly harmful or offensive content. "
        "If the text is a technical summary or a standard business email, ALLOW it."
    ),
    output_type=SafetyReviewOutput,
    model=model_registry["openai"]
)


@input_guardrail
async def guardrail_against_personal_name(ctx, _agent, user_input):
    """Block requests that include personal naming."""
    debug_print(f"DEBUG: guardrail_against_personal_name checking input: '{user_input}'")
    result = await Runner.run(name_guardrail_agent, user_input, context=ctx.context)
    name_check = result.final_output.model_dump()
    blocked = result.final_output.is_name_in_message
    debug_print(f"DEBUG: name_guardrail_agent result: blocked={blocked}, name_found='{result.final_output.name}'")
    return GuardrailFunctionOutput(
        output_info={"name_check": name_check},
        tripwire_triggered=blocked,
    )


@output_guardrail
async def outbound_safety_guardrail(ctx, _agent, output):
    """Block unsafe outbound email content."""
    debug_print("DEBUG: outbound_safety_guardrail checking generated output...")
    review = await Runner.run(output_guardrail_agent, str(output), context=ctx.context)
    safety_review = review.final_output.model_dump()
    
    blocked = review.final_output.blocked
    debug_print(f"DEBUG: outbound_safety_guardrail result: blocked={blocked}, reason='{review.final_output.reason}'")
    return GuardrailFunctionOutput(
        output_info={"safety_review": safety_review},
        tripwire_triggered=blocked,
    )


In [ ]:
sales_tools = [
    concise_sales_agent.as_tool(
        tool_name="concise_sales_agent",
        tool_description="Generate a concise-tone structured cold email.",
    ),
    engaging_sales_agent.as_tool(
        tool_name="engaging_sales_agent",
        tool_description="Generate an engaging-tone structured cold email.",
    ),
    serious_sales_agent.as_tool(
        tool_name="serious_sales_agent",
        tool_description="Generate a serious-tone structured cold email.",
    ),
    playful_sales_agent.as_tool(
        tool_name="playful_sales_agent",
        tool_description="Generate a playful-tone structured cold email.",
    ),
]

sales_manager_instructions = """
You are the Sales Manager.

You MUST follow these 5 steps in EXACT order. Do NOT stop until all steps are complete:
1. Call ALL sales agent tools (concise, engaging, serious, playful) to get candidate drafts.
2. Pass ALL generated drafts to the 'review_agent' 
tool to select the best one and get the justification.
3. Call 'get_target_contacts' to find recipients.
4. Call 'build_mail_merge_plan' using the WINNING draft's subject and content and the contacts.
5. Call 'send_mail_merge_dry_run' with the plan.

Rules:
- DO NOT finish or return a final response until you have called 'send_mail_merge_dry_run'.
- In your final response, YOU MUST include:
  - The full Subject and Body text of the chosen email draft.
  - A brief explanation of why it won.
  - The final report from the dry-run tool.
""".strip()

review_tool = review_agent.as_tool(
    tool_name="review_agent",
    tool_description="""
    Review multiple candidate drafts and select the best one. 
    Pass all candidate drafts as input to this tool.
    """,
)

manager_tools = [
    *sales_tools,
    review_tool,
    get_target_contacts,
    build_mail_merge_plan,
    send_mail_merge_dry_run,
]

sales_manager = Agent(
    name="sales_manager",
    instructions=sales_manager_instructions,
    tools=manager_tools,
    model=model_registry["gemini"],
    input_guardrails=[guardrail_against_personal_name],
    output_guardrails=[outbound_safety_guardrail],
)

In [ ]:
safe_message = "Create a SOC2 outreach email for startup CTOs and prepare a dry-run mail merge."

with trace("module2_exercise_mail_merge_guardrails"):
    safe_result = await Runner.run(sales_manager, safe_message)

print(safe_result.final_output)


In [ ]:
blocked_message = "Create and send a SOC2 outreach email addressed personally to Alice Johnson."

with trace("module2_exercise_guardrail_block_test"):
    blocked_result = await Runner.run(sales_manager, blocked_message)

print(blocked_result.final_output)
